In [1]:
import numpy as np
import torch
import torch.nn.functional as F
from utils import get_prefix_matching_scores
import plotly.graph_objects as go
from tqdm import tqdm


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-128k-instruct", use_fast=False)
model_name = "microsoft/Phi-3-mini-128k-instruct"
model = AutoModelForCausalLM.from_pretrained( 
            model_name,  
            device_map="auto",  
            torch_dtype=torch.bfloat16,  
            trust_remote_code=True,  
) 

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# Getting the scores

In [3]:
binary = {'<unk>': 0,
 '<s>': 1,
 '</s>': 2,
 '<0x00>': 3,
 '<0x01>': 4,
 '<0x02>': 5,
 '<0x03>': 6,
 '<0x04>': 7,
 '<0x05>': 8,
 '<0x06>': 9,
 '<0x07>': 10,
 '<0x08>': 11,
 '<0x09>': 12,
 '<0x0A>': 13,
 '<0x0B>': 14,
 '<0x0C>': 15,
 '<0x0D>': 16,
 '<0x0E>': 17,
 '<0x0F>': 18,
 '<0x10>': 19,
 '<0x11>': 20,
 '<0x12>': 21,
 '<0x13>': 22,
 '<0x14>': 23,
 '<0x15>': 24,
 '<0x16>': 25,
 '<0x17>': 26,
 '<0x18>': 27,
 '<0x19>': 28,
 '<0x1A>': 29,
 '<0x1B>': 30,
 '<0x1C>': 31,
 '<0x1D>': 32,
 '<0x1E>': 33,
 '<0x1F>': 34,
 '<0x20>': 35,
 '<0x21>': 36,
 '<0x22>': 37,
 '<0x23>': 38,
 '<0x24>': 39,
 '<0x25>': 40,
 '<0x26>': 41,
 '<0x27>': 42,
 '<0x28>': 43,
 '<0x29>': 44,
 '<0x2A>': 45,
 '<0x2B>': 46,
 '<0x2C>': 47,
 '<0x2D>': 48,
 '<0x2E>': 49,
 '<0x2F>': 50,
 '<0x30>': 51,
 '<0x31>': 52,
 '<0x32>': 53,
 '<0x33>': 54,
 '<0x34>': 55,
 '<0x35>': 56,
 '<0x36>': 57,
 '<0x37>': 58,
 '<0x38>': 59,
 '<0x39>': 60,
 '<0x3A>': 61,
 '<0x3B>': 62,
 '<0x3C>': 63,
 '<0x3D>': 64,
 '<0x3E>': 65,
 '<0x3F>': 66,
 '<0x40>': 67,
 '<0x41>': 68,
 '<0x42>': 69,
 '<0x43>': 70,
 '<0x44>': 71,
 '<0x45>': 72,
 '<0x46>': 73,
 '<0x47>': 74,
 '<0x48>': 75,
 '<0x49>': 76,
 '<0x4A>': 77,
 '<0x4B>': 78,
 '<0x4C>': 79,
 '<0x4D>': 80,
 '<0x4E>': 81,
 '<0x4F>': 82,
 '<0x50>': 83,
 '<0x51>': 84,
 '<0x52>': 85,
 '<0x53>': 86,
 '<0x54>': 87,
 '<0x55>': 88,
 '<0x56>': 89,
 '<0x57>': 90,
 '<0x58>': 91,
 '<0x59>': 92,
 '<0x5A>': 93,
 '<0x5B>': 94,
 '<0x5C>': 95,
 '<0x5D>': 96,
 '<0x5E>': 97,
 '<0x5F>': 98,
 '<0x60>': 99,
 '<0x61>': 100,
 '<0x62>': 101,
 '<0x63>': 102,
 '<0x64>': 103,
 '<0x65>': 104,
 '<0x66>': 105,
 '<0x67>': 106,
 '<0x68>': 107,
 '<0x69>': 108,
 '<0x6A>': 109,
 '<0x6B>': 110,
 '<0x6C>': 111,
 '<0x6D>': 112,
 '<0x6E>': 113,
 '<0x6F>': 114,
 '<0x70>': 115,
 '<0x71>': 116,
 '<0x72>': 117,
 '<0x73>': 118,
 '<0x74>': 119,
 '<0x75>': 120,
 '<0x76>': 121,
 '<0x77>': 122,
 '<0x78>': 123,
 '<0x79>': 124,
 '<0x7A>': 125,
 '<0x7B>': 126,
 '<0x7C>': 127,
 '<0x7D>': 128,
 '<0x7E>': 129,
 '<0x7F>': 130,
 '<0x80>': 131,
 '<0x81>': 132,
 '<0x82>': 133,
 '<0x83>': 134,
 '<0x84>': 135,
 '<0x85>': 136,
 '<0x86>': 137,
 '<0x87>': 138,
 '<0x88>': 139,
 '<0x89>': 140,
 '<0x8A>': 141,
 '<0x8B>': 142,
 '<0x8C>': 143,
 '<0x8D>': 144,
 '<0x8E>': 145,
 '<0x8F>': 146,
 '<0x90>': 147,
 '<0x91>': 148,
 '<0x92>': 149,
 '<0x93>': 150,
 '<0x94>': 151,
 '<0x95>': 152,
 '<0x96>': 153,
 '<0x97>': 154,
 '<0x98>': 155,
 '<0x99>': 156,
 '<0x9A>': 157,
 '<0x9B>': 158,
 '<0x9C>': 159,
 '<0x9D>': 160,
 '<0x9E>': 161,
 '<0x9F>': 162,
 '<0xA0>': 163,
 '<0xA1>': 164,
 '<0xA2>': 165,
 '<0xA3>': 166,
 '<0xA4>': 167,
 '<0xA5>': 168,
 '<0xA6>': 169,
 '<0xA7>': 170,
 '<0xA8>': 171,
 '<0xA9>': 172,
 '<0xAA>': 173,
 '<0xAB>': 174,
 '<0xAC>': 175,
 '<0xAD>': 176,
 '<0xAE>': 177,
 '<0xAF>': 178,
 '<0xB0>': 179,
 '<0xB1>': 180,
 '<0xB2>': 181,
 '<0xB3>': 182,
 '<0xB4>': 183,
 '<0xB5>': 184,
 '<0xB6>': 185,
 '<0xB7>': 186,
 '<0xB8>': 187,
 '<0xB9>': 188,
 '<0xBA>': 189,
 '<0xBB>': 190,
 '<0xBC>': 191,
 '<0xBD>': 192,
 '<0xBE>': 193,
 '<0xBF>': 194,
 '<0xC0>': 195,
 '<0xC1>': 196,
 '<0xC2>': 197,
 '<0xC3>': 198,
 '<0xC4>': 199,
 '<0xC5>': 200,
 '<0xC6>': 201,
 '<0xC7>': 202,
 '<0xC8>': 203,
 '<0xC9>': 204,
 '<0xCA>': 205,
 '<0xCB>': 206,
 '<0xCC>': 207,
 '<0xCD>': 208,
 '<0xCE>': 209,
 '<0xCF>': 210,
 '<0xD0>': 211,
 '<0xD1>': 212,
 '<0xD2>': 213,
 '<0xD3>': 214,
 '<0xD4>': 215,
 '<0xD5>': 216,
 '<0xD6>': 217,
 '<0xD7>': 218,
 '<0xD8>': 219,
 '<0xD9>': 220,
 '<0xDA>': 221,
 '<0xDB>': 222,
 '<0xDC>': 223,
 '<0xDD>': 224,
 '<0xDE>': 225,
 '<0xDF>': 226,
 '<0xE0>': 227,
 '<0xE1>': 228,
 '<0xE2>': 229,
 '<0xE3>': 230,
 '<0xE4>': 231,
 '<0xE5>': 232,
 '<0xE6>': 233,
 '<0xE7>': 234,
 '<0xE8>': 235,
 '<0xE9>': 236,
 '<0xEA>': 237,
 '<0xEB>': 238,
 '<0xEC>': 239,
 '<0xED>': 240,
 '<0xEE>': 241,
 '<0xEF>': 242,
 '<0xF0>': 243,
 '<0xF1>': 244,
 '<0xF2>': 245,
 '<0xF3>': 246,
 '<0xF4>': 247,
 '<0xF5>': 248,
 '<0xF6>': 249,
 '<0xF7>': 250,
 '<0xF8>': 251,
 '<0xF9>': 252,
 '<0xFA>': 253,
 '<0xFB>': 254,
 '<0xFC>': 255,
 '<0xFD>': 256,
 '<0xFE>': 257,
 '<0xFF>': 258,}

In [4]:
bpe_scores = {}
added_vocab = tokenizer.get_added_vocab().keys()
for k, id in tokenizer.get_vocab().items():
    if k not in added_vocab and k not in binary.keys():
        bpe_scores[k] = tokenizer.sp_model.get_score(id)
    

In [5]:
sorted_list = np.array(sorted(bpe_scores.items(), key=lambda item: item[1], reverse=True))

sorted_list[:20]

array([['▁t', '-1.0'],
       ['er', '-2.0'],
       ['in', '-3.0'],
       ['▁a', '-4.0'],
       ['en', '-5.0'],
       ['on', '-6.0'],
       ['▁th', '-7.0'],
       ['es', '-8.0'],
       ['▁s', '-10.0'],
       ['▁d', '-11.0'],
       ['at', '-12.0'],
       ['or', '-13.0'],
       ['an', '-14.0'],
       ['▁c', '-15.0'],
       ['is', '-16.0'],
       ['re', '-17.0'],
       ['it', '-18.0'],
       ['▁the', '-19.0'],
       ['ar', '-20.0'],
       ['le', '-21.0']], dtype='<U32')

In [6]:
N = len(sorted_list)
filter_percent = 0.04
filtered = sorted_list[int(N*filter_percent):int(N*(1-filter_percent))]
filtered[:20]

array([['zy', '-1278.0'],
       ['▁не', '-1279.0'],
       ['▁met', '-1280.0'],
       ['une', '-1281.0'],
       ['yth', '-1282.0'],
       ['Type', '-1283.0'],
       ['▁element', '-1284.0'],
       ['▁link', '-1285.0'],
       ['mod', '-1286.0'],
       ['▁between', '-1287.0'],
       ['cept', '-1288.0'],
       ['quire', '-1289.0'],
       ['▁through', '-1290.0'],
       ['▁while', '-1291.0'],
       ['▁On', '-1292.0'],
       ['the', '-1293.0'],
       ['ía', '-1294.0'],
       ['▁something', '-1295.0'],
       ['vol', '-1296.0'],
       ['▁most', '-1297.0']], dtype='<U32')

# Code

In [ ]:
num_seeds = 100

In [ ]:
data = {'user': 'Describe how scientific models are used to understand complex phenomena.',
 'assistant': 'Scientific models simplify systems to understand complex phenomena, predict outcomes, and test hypotheses, playing a vital role in theory development.',
 'role': 'maint_kg'}
messages = [{"role": "user", "content": data["user"]}, {"role": "assistant", "content": data["assistant"]}]
tokenized_data = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt")

In [ ]:
tokenized_data

In [ ]:
with torch.no_grad():
    output = model(input_ids = tokenized_data.to(model.device), output_attentions=True)


In [ ]:
import matplotlib.pyplot as plt
plt.plot(output.attentions[0][0,0,-10,:].float().cpu().numpy())

In [ ]:
layers = model.config.num_hidden_layers
heads = model.config.num_attention_heads
vocab = tokenizer.get_vocab()

filtered_ranks = [vocab[v[0]] for v in filtered]
prefix_matching = get_prefix_matching_scores(model, filtered_ranks, layers, heads, num_seeds=num_seeds)

torch.save(prefix_matching, "outputs/induction_heads/prefix_matching.pt")

In [5]:
prefix_matching = torch.load("outputs/induction_heads/prefix_matching.pt")

/tmp/ipykernel_3850725/837944660.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  prefix_matching = torch.load("outputs/induction_heads/prefix_matching.pt")


In [3]:
matrix = prefix_matching.mean(dim=0).cpu().numpy()
heatmap = go.Figure(data=go.Heatmap(
    z=matrix,
    x=np.arange(matrix.shape[1]),  # Column indices for x-axis
    y=np.arange(matrix.shape[0]),  # Row indices for y-axis
    colorscale='Magma',  # Set the color scale to 'Magma'
    zmin=0,  # Set the minimum value for the colormap
    zmax=1  # You can change the color scale if needed
))

# Add axis titles
heatmap.update_layout(
    title=f'Prefix Matching Scores',
    xaxis_title='Attention Heads',
    yaxis_title='Layers'
)




In [5]:
import plotly.io as pio
import os        
pio.write_html(heatmap, os.path.join(f"./outputs/induction_heads/prefix_matching.html"))

In [6]:
# Example tensor of shape (32, 32)
tensor = prefix_matching.mean(dim=0)

# Find the 99th percentile value
percentile_value = torch.quantile(tensor, 0.99)

# Create a mask that keeps values above the 99th percentile
mask = tensor >= percentile_value

# Apply the mask to the tensor
masked_tensor = tensor * mask
masked_tensor/= masked_tensor.sum()


In [7]:

heatmap = go.Heatmap(z=masked_tensor.cpu().numpy())
fig = go.Figure(data=heatmap)
fig.show()
pio.write_html(fig, os.path.join(f"./outputs/induction_heads/prefix_matching_top01.html"))

# Ablations

In [4]:
from utils import get_ablated_model
model = get_ablated_model(model)

In [29]:
tensor = prefix_matching.mean(dim=0)
percentile_value = torch.quantile(tensor, 0.99)
indices = (tensor >= percentile_value).nonzero()

layers = indices[:,0].unique()

for l in layers:
    heads = indices[:,1][indices[:,0] == l]
    model.model.layers._modules[str(l.item())].self_attn.ablate = True
    for h in heads:
        model.model.layers._modules[str(l.item())].self_attn.ablate_idx.append(h)

In [30]:
from transformers import pipeline
pipe = pipeline( 
    "text-generation", 
    model=model, 
    tokenizer=tokenizer, 
) 

generation_args = { 
    "max_new_tokens": 4096, 
    "temperature": 1.0, 
    "do_sample": False, 
} 

In [31]:
def file_to_string(filename):
    with open(filename, 'r') as file:
        return file.read()
EUREKA_ROOT_DIR="."

prompt_dir = f'./prompts'
initial_system = file_to_string(f'{prompt_dir}/initial_system.txt')
code_output_tip = file_to_string(f'{prompt_dir}/code_output_tip.txt')
initial_user = file_to_string(f'{prompt_dir}/initial_user.txt')
reward_signature = file_to_string(f'{prompt_dir}/reward_signature.txt')

tasks = {"acrobot":"to apply torques on the actuated joint to swing the free end of the linear chain above a given height from an initial state hanging downwards",
         "cartpolev1" : "to balance a pole on a cart so that the pole stays between +/- 0.209 away from the vertical",
        "cartpolev2" : "to balance a pole on a cart so that the pole stays between [0,0.21] away from the vertical",
        "orbit_discrete": "to accelerate the space shuttle to orbit the earth counter clockwise between the 2 specified bounds",
        "mountain_car": "The goal is to strategically accelerate the car to reach the goal state on top of the right hill."}

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
assistant_idx = 32001

In [32]:
def generate_text(initial_system,initial_user, model):
    messages = [
    {
        "role": "system",
        "content": initial_system,
    },
    {"role": "user", "content": initial_user},
    ]
    with torch.no_grad():
        tokenized_chat = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
        gen = model.generate(input_ids = tokenized_chat,**generation_args)[0].to("cpu")
        generated_text = tokenizer.decode(gen)
        del tokenized_chat
        torch.cuda.empty_cache()
    return generated_text, gen

In [33]:
def get_messages(task, description):
    env_name = task.replace("v1", "").replace("v2", "")
    if f'{env_name}.py' in os.listdir(f'{EUREKA_ROOT_DIR}/envs/classic_control'):
        env_parent = 'classic_control'  
    elif f'{env_name}.py' in os.listdir(f'{EUREKA_ROOT_DIR}/envs/augmented') :
        env_parent = 'augmented'
    elif f'{env_name}.py' in os.listdir(f'{EUREKA_ROOT_DIR}/envs/designed') :
        env_parent = 'designed'
    else : raise ValueError(f"Environment {env_name} not found in classic_control, designed or augmented directories!")
    task_obs_file = f'{EUREKA_ROOT_DIR}/envs/{env_parent}/{env_name}_obs.py'
    task_obs_code_string  = file_to_string(task_obs_file)
    
    system = initial_system.format(task_reward_signature_string=reward_signature)
    user = initial_user.format(task_obs_code_string=task_obs_code_string, task_description=description)
    return system, user

In [35]:
generations = {}
import os
for task, description in tqdm(tasks.items()):
    generations[task] = {}

    system, user = get_messages(task, description)

    generations[task]["text"], generations[task]["ids"] = generate_text(system, user,model)

  0%|          | 0/5 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
You are not running the flash-attention implementation, expect numerical differences.
100%|██████████| 5/5 [02:31<00:00, 30.30s/it]


In [36]:
from IPython.display import clear_output, display, Markdown
# File to read from and the new file to write annotations t            
# Display the function content
for task,gen in generations.items():
    display(Markdown(f"## {task}\n```\n{gen['text'].split('<|assistant|>')[-1]}\n```"))

## acrobot
```
 To create a reward function for the AcrobotEnv environment that encourages the agent to swing the free end of the linear chain above a given height, we need to define a clear objective for the agent. The reward function will be designed to provide positive feedback when the end of the chain is above the target height and negative feedback when it is below.

Let's assume the target height is given by `target_height`, and we have access to the current state `s` as provided by the `_get_ob` method. The reward function will be based on the vertical position of the free end of the chain, which we can calculate from the angles of the joints.

Here's a possible implementation of the reward function:

```python
import numpy as np
from typing import Tuple, Dict

def compute_reward(object_pos: np.ndarray, goal_pos: np.ndarray) -> Tuple[float, Dict[str, float]]:
    # Extract the angles and angular velocities from the state
    angle_joint_1, angle_joint_2, angular_velocity_joint_1, angular_velocity_joint_2 = object_pos
    
    # Calculate the vertical position of the free end of the chain
    # Assuming the lengths of the links are l1 and l2, and the angles are in radians
    l1, l2 = 1.0, 1.0  # Example link lengths, these should be set according to the actual environment
    vertical_position = l1 * cos(angle_joint_1) + l2 * cos(angle_joint_2)
    
    # Calculate the reward based on the vertical position
    # If the vertical position is greater than the target height, reward is positive
    # If the vertical position is less than the target height, reward is negative
    if vertical_position > goal_pos:
        reward = 1.0
    else:
        reward = -1.0
    
    # Additional information can be included in the info dictionary
    info = {}
    
    return reward, info

# Example usage:
# Assuming the target height is 1.5 units and the current state is obtained from the environment
target_height = 1.5
current_state = env.state  # This should be obtained from the environment
object_pos = env._get_ob()  # This should be a numpy array with the state information

reward, info = compute_reward(object_pos, target_height)
print(f"Reward: {reward}, Info: {info}")
```

In this example, the reward is binary, with a reward of 1.0 for achieving the goal and -1.0 otherwise. This is a simple reward function that can be used as a starting point. Depending on the specifics of the environment and the desired behavior, you might want to implement a more nuanced reward function that provides graded rewards based on how close the vertical position is to the target height.

For instance, you could use a quadratic reward function that penalizes the square of the distance from the target height, which would encourage the agent to reach the target height as quickly as possible while avoiding large deviations.

Remember to adjust the link lengths `l1` and `l2` to match the actual environment, and ensure that the `object_pos` array contains the correct state information from the environment. <|end|>
```

## cartpolev1
```
 To create a reward function for the CartPole environment that encourages the pole to remain upright within a specific angle threshold, we can use the absolute value of the pole's angle from the vertical as a measure of how well the pole is balanced. The closer the angle is to 0 (which represents the pole being perfectly upright), the higher the reward. If the pole falls outside the +/- 0.209 threshold, we can assign a large negative reward to discourage this behavior.

Here's a possible implementation of the reward function:

```python
import numpy as np
from typing import Tuple, Dict

def compute_reward(pole_angle: np.ndarray, goal_angle: float = 0.209) -> Tuple[float, Dict[str, float]]:
    """
    Reward function for the CartPole environment.

    Args:
    pole_angle (np.ndarray): The current angle of the pole from the vertical.
    goal_angle (float): The threshold angle for the pole to be considered upright.

    Returns:
    Tuple[float, Dict[str, float]]: A tuple containing the reward and a dictionary of additional information.
    """
    # Calculate the absolute difference between the current pole angle and the goal angle
    angle_diff = np.abs(pole_angle)

    # If the pole is within the goal angle threshold, give a high positive reward
    if angle_diff <= goal_angle:
        reward = 1.0
    else:
        # If the pole is outside the goal angle threshold, give a large negative reward
        reward = -1.0

    # Additional information can include the current pole angle and the goal angle
    info = {
        'current_pole_angle': pole_angle,
        'goal_angle': goal_angle
    }

    return reward, info
```

In this implementation, the `pole_angle` variable represents the current angle of the pole from the vertical, which is provided as an input to the reward function. The `goal_angle` variable is set to 0.209 by default but can be adjusted if needed.

The reward is calculated based on the absolute difference between the current pole angle and the goal angle. If the pole is within the goal angle threshold, a reward of 1.0 is given. Otherwise, a large negative reward of -1.0 is assigned to discourage the pole from falling outside the desired range.

The function returns a tuple containing the reward and a dictionary of additional information, which includes the current pole angle and the goal angle. This information can be used for further analysis or visualization of the agent's performance.

Note that this reward function assumes that the `pole_angle` input is already on the same device as the input tensors in the CartPole environment. If you're using a different environment or framework, you may need to adjust the code accordingly to ensure compatibility. <|end|>
```

## cartpolev2
```
 To create a reward function for the CartPole environment that encourages the pole to stay upright, we can use the angle of the pole (`theta`) as a key metric. The goal is to keep the pole's angle within a small range around the vertical position (0 radians). We can penalize the agent for the pole's angle deviating from this range and reward it for keeping the pole balanced.

Here's a possible implementation of the reward function:

```python
import numpy as np
from typing import Tuple, Dict

def compute_reward(object_pos: np.ndarray, goal_pos: np.ndarray) -> Tuple[float, Dict[str, float]]:
    # Extract the pole's angle from the environment's state
    theta = object_pos[2]
    
    # Define the acceptable range for the pole's angle
    theta_threshold = 0.21
    
    # Calculate the reward based on the pole's angle
    # We use a quadratic penalty for deviation from the vertical position
    # This encourages the agent to keep the pole as close to vertical as possible
    reward = -np.square(np.abs(theta - theta_threshold))
    
    # Additional information can be included in the info dictionary
    # For example, we can include the current angle of the pole
    info = {'pole_angle': theta}
    
    return reward, info
```

In this implementation, the reward is calculated as the negative square of the absolute difference between the pole's angle and the threshold (0.21 radians). This means that the closer the pole is to being vertical, the higher the reward. The penalty is quadratic, which means that larger deviations from the vertical position will result in a disproportionately higher penalty, encouraging the agent to keep the pole balanced.

The `info` dictionary can be used to provide additional information about the current state of the environment, such as the current angle of the pole. This can be useful for debugging or for visualizing the agent's behavior during training.

Note that the `object_pos` and `goal_pos` parameters are not used in this implementation, as the task only requires the pole's angle to be considered. However, if the environment provides additional information that could be useful for the reward function, such as the cart's position or velocity, those variables could be incorporated into the reward function as well. <|end|>
```

## orbit_discrete
```
 To create a reward function for the OrbitEnv environment that encourages the space shuttle to accelerate to orbit the Earth counterclockwise between the specified bounds, we need to consider the following factors:

1. The shuttle's altitude should be within the specified bounds.
2. The shuttle should be moving in the correct direction (counterclockwise) to maintain a stable orbit.
3. The shuttle should be accelerating towards the required orbital velocity.

We can define the reward function as follows:

```python
import numpy as np
from typing import Tuple, Dict

def compute_reward(state: np.ndarray, lower_bound: float, upper_bound: float) -> Tuple[float, Dict[str, float]]:
    # Extract relevant state variables
    position = state[:2]
    velocity = state[2:]
    
    # Calculate the altitude and check if it's within bounds
    altitude = np.linalg.norm(position)
    altitude_reward = max(0, lower_bound - altitude) + max(0, altitude - upper_bound)
    
    # Calculate the velocity and check if it's in the correct direction
    # Assuming the positive y-axis is the direction of the desired orbit
    # and the shuttle's velocity vector is in the same direction as the position vector
    velocity_direction = np.sign(velocity[1])
    velocity_reward = max(0, velocity_direction)
    
    # Calculate the acceleration towards the required orbital velocity
    # Assuming a constant orbital velocity for simplicity
    required_orbital_velocity = 7.8  # km/s, for example
    acceleration = np.linalg.norm(velocity) - required_orbital_velocity
    acceleration_reward = max(0, -acceleration)
    
    # Combine the rewards
    total_reward = altitude_reward + velocity_reward + acceleration_reward
    
    # Return the total reward and a dictionary of individual components
    return total_reward, {
        'altitude_reward': altitude_reward,
        'velocity_reward': velocity_reward,
        'acceleration_reward': acceleration_reward
    }

# Example usage:
# Assuming the state is a numpy array with the following format:
# [x_position, y_position, x_velocity, y_velocity]
# and the environment has been instantiated with the correct bounds
env = OrbitEnv(lower_bound=5, upper_bound=5.2)
state = env.state
reward, reward_components = compute_reward(state, env.lower_bound, env.upper_bound)
print(f"Total Reward: {reward}")
print(f"Reward Components: {reward_components}")
```

In this example, the reward function is designed to encourage the shuttle to maintain an altitude within the specified bounds, move in the correct direction, and accelerate towards the required orbital velocity. The rewards for each component are combined to form the total reward. The `compute_reward` function returns both the total reward and a dictionary containing the individual reward components for further analysis or debugging.

Please note that the actual implementation of the OrbitEnv class and the specifics of the state representation may vary, so you may need to adjust the code accordingly. Additionally, the required orbital velocity and the direction of the desired orbit should be defined based on the specific task requirements. <|end|>
```

## mountain_car
```
 To create a reward function for the MountainCarEnv environment that encourages the agent to reach the goal state on top of the right hill, we need to consider the following aspects:

1. The car's position and velocity.
2. The goal position, which is the top of the right hill.
3. The need to avoid oscillations and unnecessary movements.

Assuming that the goal position is known and fixed, we can define the reward function as follows:

```python
import numpy as np
from typing import Tuple, Dict

def compute_reward(object_pos: np.ndarray, goal_pos: np.ndarray) -> Tuple[float, Dict[str, float]]:
    """
    Reward function for MountainCarEnv.

    Args:
    object_pos (np.ndarray): The current position of the car.
    goal_pos (np.ndarray): The goal position of the car (top of the right hill).

    Returns:
    Tuple[float, Dict[str, float]]: A tuple containing the reward and a dictionary of additional information.
    """

    # Define the goal position (assuming it's known and fixed)
    goal_position = np.array([0.6, 0.05])  # Example values, adjust as needed

    # Calculate the distance to the goal
    distance_to_goal = np.linalg.norm(object_pos - goal_position)

    # Define a large negative reward for being far from the goal
    distance_penalty = -10.0 * distance_to_goal

    # Define a small positive reward for moving towards the goal
    # This encourages the car to move in the right direction
    direction_reward = 0.1 * np.dot(object_pos - goal_position, goal_pos - object_pos)

    # Define a small negative reward for unnecessary movements
    # This discourages the car from oscillating or moving in circles
    movement_penalty = -0.1 * np.linalg.norm(object_pos - np.roll(object_pos, shift=1))

    # Calculate the total reward
    reward = distance_penalty + direction_reward + movement_penalty

    # Additional information (optional)
    info = {
        'distance_to_goal': distance_to_goal,
        'direction_reward': direction_reward,
        'movement_penalty': movement_penalty
    }

    return reward, info
```

In this reward function, we have three components:

1. `distance_penalty`: A large negative reward proportional to the distance from the goal position. This encourages the agent to reach the goal state.

2. `direction_reward`: A small positive reward proportional to the dot product of the car's current position and the goal position. This encourages the car to move in the right direction.

3. `movement_penalty`: A small negative reward proportional to the norm of the difference between the current position and the previous position. This discourages the car from unnecessary movements or oscillations.

The total reward is the sum of these three components. The `info` dictionary contains additional information that can be used for debugging or analysis.

Note that the values of the rewards and penalties are hyperparameters that can be tuned to achieve better performance. The goal position (goal_position) should be adjusted based on the specific environment and task requirements. <|end|>
```

: 